# Unified Modelling Pipeline

This notebook trains a **pooled (unified) model** across all tea segments using engineered features from `tea_preprocessed_v2.csv`.

## Key Features:
1. **Engineered Features Required**: Uses `tea_preprocessed_v2.csv` as the modeling base
2. **Single Toggle**: Supply-demand features can be enabled/disabled
3. **Same 4 Models as Segment-Wise**: XGBoost, LightGBM, Gradient Boosting, Random Forest
4. **5-Fold TimeSeriesSplit CV**: Chronological ordering to prevent data leakage
5. **Segment-wise Evaluation**: Evaluate unified model performance on each segment separately

## Research Question:
What is the performance impact of adding supply-demand structure on top of an engineered-feature unified model?

**Note**: Comparison with segment-specific modeling is handled in a separate notebook (unified_vs_segment_comparison.ipynb).

## 1. Imports and Configuration

In [1]:
"""Imports and global config for unified time-series modeling."""

import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError("Please install xgboost: pip install xgboost") from exc

try:
    from lightgbm import LGBMRegressor
except ImportError as exc:
    raise ImportError("Please install lightgbm: pip install lightgbm") from exc

SEED = 42


USE_SUPPLY_DEMAND_FEATURES = False


print(f"Configuration:")
print(f"  USE_SUPPLY_DEMAND_FEATURES: {USE_SUPPLY_DEMAND_FEATURES}")
print(f"  SEED: {SEED}")
print("Libraries loaded.")

ImportError: Please install xgboost: pip install xgboost

## 2. Load Data

In [ ]:
"""Load preprocessed data with robust path detection."""

def find_repo_root(start_path: Path | None = None) -> Path:
    """Find the repository root by searching upward for the data directory."""
    candidates = []
    if start_path is not None:
        candidates.append(start_path)
    candidates.extend([Path.cwd(), Path.cwd().parent])
    for candidate in candidates:
        for root in [candidate, *candidate.parents]:
            if (root / 'data').exists() and (root / 'src').exists():
                return root
    return Path.cwd()

repo_root = find_repo_root()
processed_dir = repo_root / 'data' / 'Processed'
reports_dir = repo_root / 'reports' / 'figures'
tables_dir = repo_root / 'reports' / 'tables'

reports_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

# Base dataset is always the engineered-feature table.
base_input_path = processed_dir / 'tea_preprocessed_v2.csv'
if not base_input_path.exists():
    raise FileNotFoundError('Required engineered dataset not found: tea_preprocessed_v2.csv')

df_raw = pd.read_csv(base_input_path)
print(f'Loaded base dataset: {base_input_path.name}')
print(f'Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns')

TARGET = 'price_mid_lkr'
SEGMENT_COL = 'category_type'

df = df_raw.copy()

# Optional supply-demand enrichment from T1.3 processed dataset.
if USE_SUPPLY_DEMAND_FEATURES:
    sd_input_path = processed_dir / 'tea_preprocessed_t13_supply_demand.csv'
    sd_cols = ['supply_pressure_index', 'demand_intensity_ratio', 'market_tightness_indicator']
    if sd_input_path.exists():
        sd_df = pd.read_csv(sd_input_path)
        merge_keys = [k for k in ['sale_id', 'category_type'] if k in df.columns and k in sd_df.columns]
        if merge_keys:
            sd_keep = merge_keys + [c for c in sd_cols if c in sd_df.columns]
            sd_unique = sd_df[sd_keep].drop_duplicates(subset=merge_keys)
            df = df.merge(sd_unique, on=merge_keys, how='left')
            print(f'Enriched with supply-demand features from: {sd_input_path.name}')
        else:
            print('Warning: Could not find compatible merge keys for supply-demand enrichment.')
            USE_SUPPLY_DEMAND_FEATURES = False
    else:
        print('Warning: Supply-demand dataset not found; continuing without supply-demand features.')
        USE_SUPPLY_DEMAND_FEATURES = False
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET, SEGMENT_COL]).copy()

print(f'Rows after target/segment cleanup: {len(df):,}')
print(f'Segments: {sorted(df[SEGMENT_COL].unique())}')

Loaded base dataset: tea_preprocessed_v2.csv
Shape: 3,034 rows x 244 columns
Rows after target/segment cleanup: 2,886
Segments: ['dust', 'high_grown', 'low_grown', 'off_grade']


## 3. Feature Selection and Validation

In [ ]:
"""Detect and validate engineered and optional supply-demand features."""

# Engineered features are required in this notebook.
fe_interact = [c for c in df.columns if c.startswith('interact__')]
fe_roll = [c for c in df.columns if c.startswith('roll3_')]
fe_poly = [c for c in df.columns if c.startswith('poly2__')]

if not (fe_interact and fe_roll and fe_poly):
    raise ValueError('Engineered features are required but not found in tea_preprocessed_v2.csv')
print("Engineered features detected (required):")
print(f"  interact: {len(fe_interact)}, roll3: {len(fe_roll)}, poly2: {len(fe_poly)}")

# Build structural features for all runs
grade_rank_map = {
    "bop": 4, "bopf": 4, "bop1": 4,
    "fbop": 3, "fbop1": 3, "fbopf": 3, "fbopf1": 3, "fbopf_tippy": 3,
    "op": 2, "op1": 2, "opa": 2,
    "pekoe": 1, "pek1": 1, "pekoe_fbop": 1,
}
tier_rank_map = {"select_best": 4, "best": 3, "below_best": 2, "others": 1}
prestige_rank_map = {"high": 3, "medium": 2, "low": 1}

raw_grades = df.get("grade", pd.Series("", index=df.index)).astype(str).str.lower()
unmapped_grades = raw_grades[~raw_grades.isin(grade_rank_map) & (raw_grades != "")].unique()
if len(unmapped_grades) > 0:
    print(f"Warning: {len(unmapped_grades)} unmapped grade(s) defaulted to 0: {unmapped_grades.tolist()}")

raw_tiers = df.get("tier", pd.Series("", index=df.index)).astype(str).str.lower()
unmapped_tiers = raw_tiers[~raw_tiers.isin(tier_rank_map) & (raw_tiers != "")].unique()
if len(unmapped_tiers) > 0:
    print(f"Warning: {len(unmapped_tiers)} unmapped tier value(s) defaulted to encoded/0 fallback: {unmapped_tiers.tolist()}")

raw_prestige = df.get("elevation", pd.Series("", index=df.index)).astype(str).str.lower()
unmapped_prestige = raw_prestige[~raw_prestige.isin(prestige_rank_map) & (raw_prestige != "")].unique()
if len(unmapped_prestige) > 0:
    print(f"Warning: {len(unmapped_prestige)} unmapped elevation value(s) defaulted to 0: {unmapped_prestige.tolist()}")

df["grade_rank"] = raw_grades.map(grade_rank_map).fillna(0)
df["tier_rank"] = raw_tiers.map(tier_rank_map).fillna(df.get("tier_enc", 0)).fillna(0)
df["prestige_rank"] = raw_prestige.map(prestige_rank_map).fillna(0)

# Lagged weather variables
lag_weather_cols = [
    c for c in df.columns
    if ("lag1" in c or "lag2" in c or "lag3" in c)
    and ("precipitation" in c or "sunshine" in c or "temperature" in c)
]
print(f"Lagged weather features found: {len(lag_weather_cols)}")

# Supply-demand features (optional)
supply_demand_cols = []
if USE_SUPPLY_DEMAND_FEATURES:
    supply_demand_cols = ['supply_pressure_index', 'demand_intensity_ratio', 'market_tightness_indicator']
    supply_demand_cols = [c for c in supply_demand_cols if c in df.columns]
    if supply_demand_cols:
        print(f"Supply-demand features found: {len(supply_demand_cols)}")
    else:
        print("Warning: Supply-demand features requested but not found in dataset.")
        USE_SUPPLY_DEMAND_FEATURES = False

# Build feature list
drop_cols = {
    TARGET, 'price_mid_lkr_log', 'price_mid_usd', 'price_lo_lkr', 'price_hi_lkr',
    'price_range_lkr', 'sale_id', 'sale_date_raw', 'has_price_target',
}

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
candidate_features = [c for c in numeric_cols if c not in drop_cols]

required_features = (lag_weather_cols + ["grade_rank", "tier_rank", "prestige_rank"] + supply_demand_cols)
required_features = [c for c in required_features if c in df.columns]

# Engineered-feature setup: keep engineered expansions in the model feature set.
feature_cols = sorted(list(set(candidate_features + required_features)))

if len(feature_cols) < 5:
    raise ValueError(f"Too few features selected ({len(feature_cols)}). Check that tea_preprocessed_v2.csv contains expected engineered columns.")
print(f"Feature list validated: {len(feature_cols)} features ready for modeling.")

print(f"\nTotal features for unified modeling: {len(feature_cols)}")
print("  With engineered features: True (fixed by design)")
print(f"  With supply-demand features: {USE_SUPPLY_DEMAND_FEATURES}")

Engineered features detected (required):
  interact: 28, roll3: 19, poly2: 15
Lagged weather features found: 36

Total features for unified modeling: 231
  With engineered features: True (fixed by design)
  With supply-demand features: False


## 4. Data Preparation for Unified Modeling

In [ ]:
"""Prepare data for pooled (unified) modeling with chronological sorting."""

# Sort chronologically by auction order
sort_cols = [c for c in ["sale_year", "sale_number"] if c in df.columns]
if sort_cols:
    df = df.sort_values(sort_cols).reset_index(drop=True)
else:
    print("Warning: Could not find sale_year/sale_number for chronological sorting.")

# Extract features and target
X = df[feature_cols].copy()
y = df[TARGET].copy()

# Ensure target is numeric and remove NaN
y = pd.to_numeric(y, errors='coerce')
valid_mask = y.notna()
X = X[valid_mask].reset_index(drop=True)
y = y[valid_mask].reset_index(drop=True)
df_supervised = df[valid_mask].reset_index(drop=True)

print(f"Unified dataset prepared:")
print(f"  Total rows: {len(X):,}")
print(f"  Features: {len(feature_cols)}")
print(f"  Target shape: {y.shape}")
print(f"\nSegment distribution:")
for seg in sorted(df_supervised[SEGMENT_COL].unique()):
    seg_rows = (df_supervised[SEGMENT_COL] == seg).sum()
    print(f"  {seg}: {seg_rows:,} rows")

Unified dataset prepared:
  Total rows: 2,886
  Features: 231
  Target shape: (2886,)

Segment distribution:
  dust: 523 rows
  high_grown: 516 rows
  low_grown: 1,348 rows
  off_grade: 499 rows


## 5. Model Definitions (Same as Segment-Wise Pipeline)

In [ ]:
"""Define the same 4 models as segment-wise modeling for direct comparison."""

models = {
    "XGBoost": XGBRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,  # random_state controls all model-level reproducibility.
        n_jobs=-1,
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=-1,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
    ),
}

def compute_metrics(y_true, y_pred):
    """Compute RMSE, MAE, MAPE, and R2 on raw LKR prices."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    # Clip denominator to 1e-9 to avoid division by zero; may inflate MAPE if true prices are near zero.
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-9))) * 100
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, mape, r2

print(f"Models defined: {list(models.keys())}")

Models defined: ['XGBoost', 'LightGBM', 'Gradient Boosting', 'Random Forest']


## 6. Unified Model Training with 5-Fold TimeSeriesSplit CV

In [ ]:
"""Train unified model with 5-fold TimeSeriesSplit cross-validation."""

print("Running 5-fold TimeSeriesSplit on UNIFIED (pooled) data across all segments...\n")

# Final chronological holdout: last 15% of rows are kept untouched for one-time final evaluation.
holdout_size = max(1, int(np.ceil(len(X) * 0.15)))
train_size = len(X) - holdout_size
if train_size <= 5:
    raise ValueError(f"Not enough rows ({len(X)}) to create 85/15 split with 5-fold CV.")

X_train_cv = X.iloc[:train_size].reset_index(drop=True)
y_train_cv = y.iloc[:train_size].reset_index(drop=True)
df_train_cv = df_supervised.iloc[:train_size].reset_index(drop=True)

X_holdout = X.iloc[train_size:].reset_index(drop=True)
y_holdout = y.iloc[train_size:].reset_index(drop=True)
df_holdout = df_supervised.iloc[train_size:].reset_index(drop=True)

print("Chronological split completed:")
print(f"  CV train rows (first 85%): {len(X_train_cv):,}")
print(f"  Holdout rows (last 15%): {len(X_holdout):,}")

tscv = TimeSeriesSplit(n_splits=5)
all_unified_fold_results = []
oof_preds_by_model = {model_name: np.full(len(y_train_cv), np.nan) for model_name in models}

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_train_cv), start=1):
    X_train, X_test = X_train_cv.iloc[train_idx], X_train_cv.iloc[test_idx]
    y_train, y_test = y_train_cv.iloc[train_idx], y_train_cv.iloc[test_idx]

    for model_name, model in models.items():
        pipe = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", clone(model)),
        ])
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        rmse, mae, mape, r2 = compute_metrics(y_test, preds)

        # Store out-of-fold predictions for leakage-free segment-wise evaluation later.
        oof_preds_by_model[model_name][test_idx] = preds

        all_unified_fold_results.append({
            "Fold": fold_idx,
            "Model": model_name,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape,
            "R2": r2,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
        })

unified_fold_results = pd.DataFrame(all_unified_fold_results)

# Compute fold-level summary
unified_summary = (
    unified_fold_results.groupby("Model", as_index=False)
    .agg(
        RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
        MAPE_mean=("MAPE", "mean"), MAPE_std=("MAPE", "std"),
        R2_mean=("R2", "mean"), R2_std=("R2", "std"),
    )
    .sort_values("RMSE_mean")
    .reset_index(drop=True)
)

print("Unified Model 5-Fold CV Summary (sorted by RMSE):")
display(unified_summary.round(4))

# Build an all-model findings table with ranks and stability indicators.
unified_all_model_findings = unified_summary.copy()
unified_all_model_findings['RMSE_rank'] = unified_all_model_findings['RMSE_mean'].rank(method='dense', ascending=True).astype(int)
unified_all_model_findings['R2_rank'] = unified_all_model_findings['R2_mean'].rank(method='dense', ascending=False).astype(int)
unified_all_model_findings['RMSE_cv_pct'] = (unified_all_model_findings['RMSE_std'] / unified_all_model_findings['RMSE_mean']) * 100.0
unified_all_model_findings['R2_cv_pct'] = (unified_all_model_findings['R2_std'].abs() / unified_all_model_findings['R2_mean'].abs().replace(0, np.nan)) * 100.0
unified_all_model_findings = unified_all_model_findings.sort_values(['RMSE_rank', 'R2_rank']).reset_index(drop=True)

print('All-Model Findings Table (not limited to best model):')
display(unified_all_model_findings.round(4))

Running 5-fold TimeSeriesSplit on UNIFIED (pooled) data across all segments...

Unified Model 5-Fold CV Summary (sorted by RMSE):


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std
0,Random Forest,221.7360,44.0899,135.0704,19.4967,12.1811,1.4848,0.8728,0.0535
1,Gradient Boosting,242.2629,21.9503,158.2122,12.6109,14.5789,1.4819,0.8515,0.0292
2,LightGBM,272.0150,84.0514,154.4827,29.6985,13.4946,1.5163,0.8014,0.1184
3,XGBoost,284.6029,62.6681,152.9033,16.8252,13.2463,0.9992,0.7917,0.0834


All-Model Findings Table (not limited to best model):


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,RMSE_rank,R2_rank,RMSE_cv_pct,R2_cv_pct
0,Random Forest,221.7360,44.0899,135.0704,19.4967,12.1811,1.4848,0.8728,0.0535,1,1,19.8839,6.1259
1,Gradient Boosting,242.2629,21.9503,158.2122,12.6109,14.5789,1.4819,0.8515,0.0292,2,2,9.0605,3.4323
2,LightGBM,272.0150,84.0514,154.4827,29.6985,13.4946,1.5163,0.8014,0.1184,3,3,30.8996,14.7777
3,XGBoost,284.6029,62.6681,152.9033,16.8252,13.2463,0.9992,0.7917,0.0834,4,4,22.0195,10.5316


## 7. Segment-Wise Evaluation of Unified Model

In [ ]:
"""Evaluate unified model performance on each segment separately."""

# Select best model via composite rank: lower RMSE and higher R2, tie-broken by lower RMSE_std.
model_selection_table = unified_summary.copy()
model_selection_table['RMSE_rank'] = model_selection_table['RMSE_mean'].rank(method='dense', ascending=True).astype(int)
model_selection_table['R2_rank'] = model_selection_table['R2_mean'].rank(method='dense', ascending=False).astype(int)
model_selection_table['combined_rank'] = model_selection_table['RMSE_rank'] + model_selection_table['R2_rank']
model_selection_table = model_selection_table.sort_values(['combined_rank', 'RMSE_std', 'RMSE_mean']).reset_index(drop=True)

print('Best-model selection table (composite rank with stability tie-break):')
display(model_selection_table.round(4))

best_model_name = model_selection_table.iloc[0]['Model']
best_model_config = models[best_model_name]

# Leakage-free segment evaluation uses out-of-fold predictions from CV train portion only.
best_oof_preds = oof_preds_by_model[best_model_name]
if np.isnan(best_oof_preds).any():
    raise ValueError(f"OOF predictions for {best_model_name} contain missing values. Check CV setup.")

df_train_cv_eval = df_train_cv.copy()
df_train_cv_eval['unified_pred_lkr'] = best_oof_preds
df_train_cv_eval['unified_residual'] = df_train_cv_eval[TARGET] - df_train_cv_eval['unified_pred_lkr']
df_train_cv_eval['unified_abs_error'] = np.abs(df_train_cv_eval['unified_residual'])

# Evaluate by segment using OOF predictions
unified_segment_results = []
for segment in sorted(df_train_cv_eval[SEGMENT_COL].unique()):
    seg_mask = df_train_cv_eval[SEGMENT_COL] == segment
    seg_y_true = df_train_cv_eval.loc[seg_mask, TARGET].values
    seg_y_pred = df_train_cv_eval.loc[seg_mask, 'unified_pred_lkr'].values

    if len(seg_y_true) == 0:
        continue

    rmse, mae, mape, r2 = compute_metrics(seg_y_true, seg_y_pred)
    unified_segment_results.append({
        'Segment': segment,
        'Model': 'Unified (OOF)',
        'N_Rows': len(seg_y_true),
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape,
        'R2': r2,
    })

unified_by_segment = pd.DataFrame(unified_segment_results)

print(f"\nBest Unified Model: {best_model_name}")
print("\nUnified Model OOF Performance by Segment (CV train portion):")
display(unified_by_segment.round(4))

# Final holdout evaluation: retrain best model on full 85% train portion, then test on last 15%.
pipe_holdout = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('model', clone(best_model_config)),
])
pipe_holdout.fit(X_train_cv, y_train_cv)
holdout_preds = pipe_holdout.predict(X_holdout)
holdout_rmse, holdout_mae, holdout_mape, holdout_r2 = compute_metrics(y_holdout.values, holdout_preds)

unified_holdout_evaluation = pd.DataFrame([
    {
        'Model': best_model_name,
        'Train_Rows': len(X_train_cv),
        'Holdout_Rows': len(X_holdout),
        'RMSE': holdout_rmse,
        'MAE': holdout_mae,
        'MAPE': holdout_mape,
        'R2': holdout_r2,
    }
])

print("\nFinal Chronological Holdout Evaluation (last 15%):")
display(unified_holdout_evaluation.round(4))

# Train final deployment model on all available data (do not use this fit for evaluation metrics).
pipe_best = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('model', clone(best_model_config)),
])
pipe_best.fit(X, y)
print("Final deployment model trained on full dataset.")


Best Unified Model: Random Forest

Unified Model Performance by Segment:


,Segment,Model,N_Rows,RMSE,MAE,MAPE,R2
0,dust,Unified,523,52.4621,35.1707,4.0791,0.9358
1,high_grown,Unified,516,77.6505,54.0575,6.1094,0.9116
2,low_grown,Unified,1348,49.2424,25.7672,1.5884,0.9957
3,off_grade,Unified,499,48.4936,34.1643,4.8534,0.9178


## 8. Save Results

In [ ]:
"""Save unified model results to disk."""

config_suffix = "_with_supply_demand" if USE_SUPPLY_DEMAND_FEATURES else "_no_supply_demand"

unified_fold_results.to_csv(
    tables_dir / f"unified_cv_fold_results{config_suffix}.csv",
    index=False,
)
unified_summary.to_csv(
    tables_dir / f"unified_cv_summary{config_suffix}.csv",
    index=False,
)
unified_all_model_findings.to_csv(
    tables_dir / f"unified_all_model_findings{config_suffix}.csv",
    index=False,
)
unified_by_segment.to_csv(
    tables_dir / f"unified_segment_evaluation{config_suffix}.csv",
    index=False,
)
unified_holdout_evaluation.to_csv(
    tables_dir / f"unified_holdout_evaluation{config_suffix}.csv",
    index=False,
)

print("Saved:")
print(f"- unified_cv_fold_results{config_suffix}.csv")
print(f"- unified_cv_summary{config_suffix}.csv")
print(f"- unified_all_model_findings{config_suffix}.csv")
print(f"- unified_segment_evaluation{config_suffix}.csv")
print(f"- unified_holdout_evaluation{config_suffix}.csv")

Saved:
- unified_cv_fold_results_no_supply_demand.csv
- unified_cv_summary_no_supply_demand.csv
- unified_all_model_findings_no_supply_demand.csv
- unified_segment_evaluation_no_supply_demand.csv


## 9. Configuration Summary and Results

In [ ]:
# Save config metadata for report-ready outputs.
tables_dir.mkdir(parents=True, exist_ok=True)

config_path = tables_dir / f"unified_pipeline_config{config_suffix}.json"
config_payload = {
    "engineered_feature_source": "tea_preprocessed_v2.csv",
    "use_engineered_features": True,
    "use_supply_demand_features": "with_supply_demand" in config_suffix,
    "config_suffix": config_suffix,
}
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=2)

print("Saved report-ready outputs:")
print(f"- unified_pipeline_config{config_suffix}.json")

Saved report-ready outputs:
- unified_cv_summary_no_supply_demand.csv
- unified_all_model_findings_no_supply_demand.csv
- unified_segment_evaluation_no_supply_demand.csv
- unified_pipeline_config_no_supply_demand.json
